# Patrón de Comportamiento: Iterator

## Introducción
El patrón Iterator proporciona una forma de acceder secuencialmente a los elementos de un objeto agregado sin exponer su representación interna.

## Objetivos
- Comprender cómo recorrer colecciones sin exponer su estructura interna.
- Identificar cuándo es útil el patrón Iterator.
- Comparar la solución con y sin el patrón.

## Ejemplo de la vida real
**Contexto: Reproductor de música**
Un reproductor de música recorre listas de canciones, permitiendo avanzar, retroceder o repetir sin exponer la estructura interna de la lista.

**¿Dónde se usa en proyectos reales?**
En colecciones, bases de datos, sistemas de archivos, etc.

## Sin patrón Iterator (forma errónea)
El cliente accede directamente a la estructura interna de la colección.

In [1]:
class Playlist:
    def __init__(self, canciones):
        self.canciones = canciones

playlist = Playlist(['A', 'B', 'C'])
for i in range(len(playlist.canciones)):
    print(playlist.canciones[i])

A
B
C


## Con patrón Iterator (forma correcta)
El cliente usa un iterador para recorrer la colección.

In [2]:
class IteradorPlaylist:
    def __init__(self, playlist):
        self._playlist = playlist
        self._indice = 0
    def __next__(self):
        if self._indice < len(self._playlist.canciones):
            cancion = self._playlist.canciones[self._indice]
            self._indice += 1
            return cancion
        else:
            raise StopIteration
    def __iter__(self):
        return self

class Playlist:
    def __init__(self, canciones):
        self.canciones = canciones
    def __iter__(self):
        return IteradorPlaylist(self)

playlist = Playlist(['A', 'B', 'C'])
for cancion in playlist:
    print(cancion)

A
B
C


## UML del patrón Iterator
```plantuml
@startuml
class Playlist {
    + __iter__()
}
class IteradorPlaylist {
    + __next__()
    + __iter__()
}
Playlist --> IteradorPlaylist
@enduml
```

## Otro ejemplo de la vida real: Paginación de resultados de una API/base de datos
**Contexto:** cuando una tabla tiene millones de registros, no tiene sentido cargarlos todos en memoria de una vez. APIs y ORMs recorren los resultados **página por página** (paginación por cursor), trayendo la siguiente página desde la base de datos solo cuando el cliente la necesita.

### Sin patrón (forma errónea)
Se asume que todos los registros ya están cargados en memoria, y el cliente calcula manualmente los índices de cada página.

In [3]:
class TablaUsuarios:
    def __init__(self, registros):
        self.registros = registros  # simula que TODOS los registros ya están en memoria

tabla = TablaUsuarios([f'usuario_{i}' for i in range(10)])
pagina = 0
tamano_pagina = 3
inicio = pagina * tamano_pagina
for r in tabla.registros[inicio:inicio + tamano_pagina]:
    print(r)
# El cliente calcula índices a mano, y en un caso real la tabla completa ya estaría en memoria

usuario_0
usuario_1
usuario_2


### Con patrón (forma correcta)
`IteradorPaginado` trae una página nueva de la "base de datos" solo cuando se acaba el buffer actual. El cliente simplemente hace `for usuario in ...`, sin saber que por debajo hay páginas ni consultas.

In [4]:
class IteradorPaginado:
    def __init__(self, obtener_pagina, tamano_pagina=3):
        self._obtener_pagina = obtener_pagina
        self._tamano_pagina = tamano_pagina
        self._pagina_actual = 0
        self._buffer = []
    def __iter__(self):
        return self
    def __next__(self):
        if not self._buffer:
            self._buffer = self._obtener_pagina(self._pagina_actual, self._tamano_pagina)
            self._pagina_actual += 1
            if not self._buffer:
                raise StopIteration
        return self._buffer.pop(0)


def consultar_base_datos(pagina, tamano):
    todos = [f'usuario_{i}' for i in range(10)]
    inicio = pagina * tamano
    resultado = todos[inicio:inicio + tamano]
    if resultado:
        print(f'-- Consultando página {pagina} a la base de datos --')
    return resultado


for usuario in IteradorPaginado(consultar_base_datos, tamano_pagina=3):
    print(usuario)

-- Consultando página 0 a la base de datos --
usuario_0
usuario_1
usuario_2
-- Consultando página 1 a la base de datos --
usuario_3
usuario_4
usuario_5
-- Consultando página 2 a la base de datos --
usuario_6
usuario_7
usuario_8
-- Consultando página 3 a la base de datos --
usuario_9


### UML del ejemplo de paginación
```plantuml
@startuml
class IteradorPaginado {
    - _obtener_pagina
    - _tamano_pagina
    - _pagina_actual
    - _buffer: list
    + __iter__()
    + __next__()
}
@enduml
```

### ¿Dónde más se usa Iterator?
- **Paginación de APIs/bases de datos:** exactamente este ejemplo — cursores de MongoDB, `Paginator` de Django, scroll de Elasticsearch.
- **Generadores de Python:** cualquier función con `yield` es, por debajo, una implementación del patrón Iterator (produce valores uno a la vez, bajo demanda).
- **Recorrido de árboles/grafos:** iterar los nodos de un árbol (in-order, pre-order) sin que el cliente conozca la estructura interna del árbol.
- **Streaming de archivos grandes:** leer un archivo línea por línea o en bloques, sin cargarlo completo en memoria.
- **Reproductores multimedia:** el ejemplo con el que abre este notebook — recorrer una playlist sin exponer la lista interna de canciones.

**Ejercicio de reflexión:** ¿cómo modificarías `IteradorPaginado` para que, si `consultar_base_datos` devuelve la misma página dos veces seguidas por un error de red, no se quede en un ciclo infinito?

## Actividad
Crea un iterador para recorrer los asientos de un avión, permitiendo avanzar y retroceder.

---
## Explicación de conceptos clave
- **Abstracción de recorrido:** El cliente no conoce la estructura interna.
- **Flexibilidad:** Se pueden crear diferentes tipos de iteradores.
- **Aplicación en la vida real:** Útil en colecciones, bases de datos y sistemas de archivos.

## Conclusión
El patrón Iterator es ideal para recorrer colecciones de manera flexible y segura.